
# GENIE.AI Ingestion Notebook (Colab)

End-to-end, production-style notebook for converting Bangladesh climate/agriculture CSV outputs into GENIE.AI ingestion artifacts.

## Pipeline Steps
1. Install dependencies  
2. Load all CSV files  
3. Convert CSV to Markdown documents  
4. Prepare ArangoDB JSON documents  
5. Prepare BigQuery schema + ingestion-ready tables  
6. Upload to GENIE.AI (SDK examples)  
7. Test retrieval


## Step 1: Install dependencies

In [ ]:

# Colab dependency setup
!pip -q install pandas pyarrow google-cloud-bigquery pandas-gbq abacusai


## Step 2: Load all CSV files

In [ ]:

from pathlib import Path
import pandas as pd

# Update if your CSVs are in a different folder
INPUT_DIR = Path('/content/uploads')
# Alternative defaults often used in this project:
# INPUT_DIR = Path('/home/ubuntu/Uploads')

csv_paths = sorted(INPUT_DIR.glob('*.csv'))
if not csv_paths:
    raise FileNotFoundError(f'No CSV files found in {INPUT_DIR}')

frames = {}
for p in csv_paths:
    df = pd.read_csv(p)
    frames[p.name] = df
    print(f'{p.name}: rows={len(df):,}, cols={len(df.columns)}')

print(f"\nLoaded {len(frames)} CSV files")


## Step 3: Convert CSV files to Markdown documents

In [ ]:

import sys
from pathlib import Path

# Make sure converter module is available in Colab runtime
# Option A: upload production_pipeline folder to /content first
PROJECT_ROOT = Path('/content')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from production_pipeline.genie_data_converter import GenieDataConverter

OUTPUT_DIR = Path('/content/genie_outputs')
converter = GenieDataConverter(input_dir=INPUT_DIR, output_dir=OUTPUT_DIR)
manifest = converter.convert_all()
manifest


In [ ]:

# Quick sanity check of generated markdown docs
md_files = sorted((OUTPUT_DIR / 'markdown_docs').glob('**/*.md'))
print(f'Generated markdown docs: {len(md_files)}')
for p in md_files[:10]:
    print('-', p)


## Step 4: Prepare ArangoDB format (JSON documents)

In [ ]:

import json
from pathlib import Path

meta_dir = OUTPUT_DIR / 'metadata'
arango_jsonl = meta_dir / 'arango_documents.jsonl'
metadata_jsonl = meta_dir / 'documents_metadata.jsonl'

print('Arango JSONL:', arango_jsonl)
print('Metadata JSONL:', metadata_jsonl)

# Optional: Load a few example docs
sample_docs = []
with arango_jsonl.open('r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        sample_docs.append(json.loads(line))
        if i >= 2:
            break

sample_docs


In [ ]:

# Optional: export collections for arango import CLI usage
# arangoimport --file arango_documents.jsonl --type jsonl --collection GRAPH_source_documents

import shutil
shutil.copy('/content/production_pipeline/arango_schema.json', OUTPUT_DIR / 'arango_schema.json')
print('Copied arango_schema.json into output directory.')


## Step 5: Prepare BigQuery schema and data

In [ ]:

# Build ingestion-ready tables from metadata for BigQuery load jobs
import json
import pandas as pd

metadata_path = OUTPUT_DIR / 'metadata' / 'documents_metadata.jsonl'
meta_df = pd.read_json(metadata_path, lines=True)

# Base timestamps/dates
meta_df['record_date'] = pd.to_datetime(meta_df['record_date'], errors='coerce').dt.date
meta_df['created_at_utc'] = pd.to_datetime(meta_df['created_at_utc'], errors='coerce')

# 1) drought_monitoring
drought_df = meta_df[meta_df['category_label'] == 'Drought Monitoring'].copy()

# 2) rainfall_climate
rain_climate_df = meta_df[meta_df['category_label'] == 'Rainfall & Climate'].copy()

# 3) agriculture_monitoring
agri_df = meta_df[meta_df['category_label'] == 'Agriculture'].copy()

# 4) district_profiles
district_profiles_df = meta_df[meta_df['category_label'] == 'District Profiles'].copy()

print('drought_monitoring rows:', len(drought_df))
print('rainfall_climate rows:', len(rain_climate_df))
print('agriculture_monitoring rows:', len(agri_df))
print('district_profiles rows:', len(district_profiles_df))


In [ ]:

# Save BigQuery-ready CSV artifacts
bq_dir = OUTPUT_DIR / 'bigquery_ready'
bq_dir.mkdir(parents=True, exist_ok=True)

drought_df.to_csv(bq_dir / 'drought_monitoring.csv', index=False)
rain_climate_df.to_csv(bq_dir / 'rainfall_climate.csv', index=False)
agri_df.to_csv(bq_dir / 'agriculture_monitoring.csv', index=False)
district_profiles_df.to_csv(bq_dir / 'district_profiles.csv', index=False)

print('Saved BigQuery-ready CSVs to', bq_dir)


In [ ]:

# Optional: load into BigQuery (requires GCP auth in Colab)
# from google.cloud import bigquery
# client = bigquery.Client(project='YOUR_PROJECT_ID')
# dataset = 'bangladesh_indicators'
#
# def load_csv(csv_path, table_name):
#     job_config = bigquery.LoadJobConfig(
#         source_format=bigquery.SourceFormat.CSV,
#         skip_leading_rows=1,
#         autodetect=True,
#         write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
#     )
#     with open(csv_path, 'rb') as f:
#         job = client.load_table_from_file(f, f'{client.project}.{dataset}.{table_name}', job_config=job_config)
#     job.result()
#     print(f'Loaded {job.output_rows} rows -> {table_name}')
#
# load_csv(bq_dir / 'drought_monitoring.csv', 'drought_monitoring')
# load_csv(bq_dir / 'rainfall_climate.csv', 'rainfall_climate')
# load_csv(bq_dir / 'agriculture_monitoring.csv', 'agriculture_monitoring')
# load_csv(bq_dir / 'district_profiles.csv', 'district_profiles')


## Step 6: Upload to GENIE.AI (SDK examples)

In [ ]:

import abacusai
from pathlib import Path

client = abacusai.ApiClient()

# Discover relevant APIs dynamically (safe when SDK signatures differ by version)
apis = client.suggest_abacus_apis('create document retriever and ingest markdown documents for RAG')
apis


In [ ]:

# Example ingestion flow (fill in IDs from your workspace)
# 1) Create / use a document retriever
# retriever = client.create_document_retriever(project_id='YOUR_PROJECT_ID', name='bd-climate-retriever')
#
# 2) Upload markdown files to a dataset or storage location
# md_paths = [str(p) for p in (OUTPUT_DIR / 'markdown_docs').glob('**/*.md')]
#
# 3) Ingest docs with labels metadata
# (Exact method names can differ; use suggest_abacus_apis output above.)
#
# for p in md_paths[:5]:
#     print('Would ingest:', p)


## Step 7: Test retrieval

In [ ]:

# Retrieval smoke test example (adapt to your deployment)
# deployment_id = 'YOUR_DEPLOYMENT_ID'
# deployment = client.describe_deployment(deployment_id)
# tokens = client.list_deployment_tokens(deployment.project_id)
# deployment_token = tokens[0].deployment_token if tokens else client.create_deployment_token(deployment.project_id).deployment_token
#
# response = client.get_chat_response(
#     deployment_token=deployment_token,
#     deployment_id=deployment_id,
#     messages=[{'role': 'user', 'content': 'What is the latest drought status for Dhaka district?'}]
# )
# print(response)



## Outputs generated by this notebook
- `genie_outputs/markdown_docs/**` (GENIE ingestion documents)
- `genie_outputs/metadata/documents_metadata.jsonl`
- `genie_outputs/metadata/arango_documents.jsonl`
- `genie_outputs/bigquery_ready/*.csv`

Use `production_pipeline/bigquery_schema.sql` and `production_pipeline/arango_schema.json` for infrastructure provisioning.
